# Candidate-retrieval precision analysis

Compute precision at K from saved model similarities and ground-truth Tanimoto / normalized MCES similarity matrices, then generate the four comparison figures. The query and reference filtering must match the ordering used when producing the saved results.

## Configuration

In [ ]:
import os
import sys
from pathlib import Path
import copy
import dill
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.collections import LineCollection
from rdkit import Chem

from simba.config import Config
from simba.load_data import LoadData
from simba.preprocessor import Preprocessor
from simba.analog_discovery.performance_analysis import PerformanceAnalysis
import simba

# Compatibility alias needed for loading older serialized data.
sys.modules["src"] = simba

PROJECT_ROOT = Path("/Users/sebas/projects/metabolomics")
DATA_FOLDER = Path("/Users/sebas/projects/data")
FIGURE_FOLDER = Path("/Users/sebas/Documents/paper_simba")
RESULT_FOLDER = PROJECT_ROOT / "notebooks_local/discovery_search/results"

USE_CASMI = True
USE_GNPS = False
TARGET_SIMBA = "simba_reduce_sim1_v3"

DATASET_PATH = DATA_FOLDER / "edit_distance_neurips_nist_exhaustive.pkl"
CASMI_FILE = DATA_FOLDER / "processed_massformer/spec_df.pkl"
GNPS_FILE = DATA_FOLDER / "ALL_GNPS_NO_PROPOGATED_wb.mgf"

# Janssen branch (optional): configure the path and LoaderSaver import below before use.
OUTPUT_JANSSEN_FILE = DATA_FOLDER / "all_spectrums_janssen.pkl"
JANSSEN_PATH = None

MODEL_NAMES = [
    "modified_cosine_casmi_NEURIPS",
    "spec2vec_casmi_NEURIPS",
    "ms2deepscore_v2_casmi_NEURIPS",
    "dreams_casmi_NEURIPS",
    "simba_20250422_only_massspecgym",
    TARGET_SIMBA,
]
GNPS_MODEL_NAMES = [
    "modified_cosine_casmi_NEURIPS_gnps",
    "spec2vec_casmi_NEURIPS_gnps",
    "ms2deepscore_v2_casmi_NEURIPS_gnps",
    "dreams_casmi_NEURIPS_gnps",
    "simba_generated_data_20250130_replicated_gnps",
]
MODEL_NAMES = GNPS_MODEL_NAMES if USE_GNPS else MODEL_NAMES
THRESHOLD_MODEL = GNPS_MODEL_NAMES[-1] if USE_GNPS else TARGET_SIMBA
SUFFIX = "_gnps" if USE_GNPS else ""
TANIMOTO_NAME = "tanimoto_similarity_casmi_NEURIPS" + SUFFIX
MCES_NAME = "mces_similarity_casmi_NEURIPS" + SUFFIX

MIN_SIMILARITIES = [0, 0.85, 0.9, 0.95]
K_VALUES = [5, 10, 20, 40, 80, 100]
FIGURE_FOLDER.mkdir(parents=True, exist_ok=True)


def load_result(name):
    path = RESULT_FOLDER / f"{name}_results_analog_discovery_unknown_compounds.pkl"
    with path.open("rb") as file:
        return dill.load(file)

## Load and filter reference / query spectra

In [ ]:
config = Config()
preprocessor = Preprocessor()

if USE_GNPS:
    from simba.simba.preprocessing_simba import PreprocessingSimba
    reference_original = PreprocessingSimba.load_spectra(
        str(GNPS_FILE), config, use_gnps_format=True
    )
    reference_original = [s for s in reference_original if Chem.MolFromSmiles(s.smiles) is not None]
else:
    with DATASET_PATH.open("rb") as file:
        dataset = dill.load(file)
    reference_original = (
        dataset["molecule_pairs_train"].spectrums_original
        + dataset["molecule_pairs_val"].spectrums_original
        + dataset["molecule_pairs_test"].spectrums_original
    )

def filter_spectra(spectra):
    # Preprocess copies only for filtering; retain original spectra for metric computation.
    filtered = []
    for original in spectra:
        processed = preprocessor.preprocess_spectrum(
            copy.deepcopy(original),
            fragment_tol_mass=10,
            fragment_tol_mode="ppm",
            min_intensity=0.01,
            max_num_peaks=100,
            scale_intensity=None,
        )
        if len(processed.mz) >= 6:
            filtered.append(original)
    return filtered

all_spectrums_reference = filter_spectra(reference_original)

if USE_CASMI:
    query_original = LoadData.get_all_spectrums_casmi(str(CASMI_FILE), config=config)
else:
    if JANSSEN_PATH is None:
        raise ValueError("Set JANSSEN_PATH and supply the LoaderSaver import for USE_CASMI=False")
    # Adjust this import if LoaderSaver lives elsewhere in your project.
    from simba.load_data import LoaderSaver
    loader = LoaderSaver(
        block_size=100, pickle_nist_path="", pickle_gnps_path="",
        pickle_janssen_path=str(OUTPUT_JANSSEN_FILE),
    )
    query_original = loader.get_all_spectrums(
        JANSSEN_PATH, 100000000, use_tqdm=True, use_nist=False,
        config=config, use_janssen=True,
    )

queries_filtered = filter_spectra(query_original)
reference_smiles = {Chem.CanonSmiles(s.smiles) for s in all_spectrums_reference}
all_spectrums_janssen = [
    s for s in queries_filtered if Chem.CanonSmiles(s.smiles) not in reference_smiles
]
print(f"Reference spectra: {len(all_spectrums_reference):,}; queries: {len(all_spectrums_janssen):,}")

## Load saved similarities and compute precision at K

In [ ]:
model_results = {name: load_result(name) for name in MODEL_NAMES}
ground_truth = {
    "tanimotos": load_result(TANIMOTO_NAME)["similarities"],
    "mces": load_result(MCES_NAME)["similarities"],
}

accuracy_results = {}
for metric, truth in ground_truth.items():
    accuracy_results[metric] = {threshold: {} for threshold in MIN_SIMILARITIES}
    for model_name, model_data in model_results.items():
        precision = PerformanceAnalysis.obtain_precision_k_candidates(
            golden_truth=truth,
            predicted=model_data["similarities"],
            smiles_reference=model_data["smiles_reference"],
            smiles_janssen=model_data["smiles_janssen"],
            all_spectrums_reference=all_spectrums_reference,
            all_spectrums_janssen=all_spectrums_janssen,
            min_tanimotos=MIN_SIMILARITIES,
            k_list=K_VALUES,
        )
        for threshold in MIN_SIMILARITIES:
            accuracy_results[metric][threshold][model_name] = precision[threshold]

## Figures

In [ ]:
MODEL_LABELS = {
    "modified_cosine_casmi_NEURIPS": "Modified cosine",
    "spec2vec_casmi_NEURIPS": "Spec2Vec",
    "ms2deepscore_v2_casmi_NEURIPS": "MS2DeepScore",
    "dreams_casmi_NEURIPS": "DreaMS",
    "simba_20250422_only_massspecgym": "SIMBA (MassSpecGym)",
    TARGET_SIMBA: "SIMBA",
}
# Display labels for GNPS models, in their configured order.
MODEL_LABELS.update(dict(zip(GNPS_MODEL_NAMES, [
    "Modified cosine", "Spec2Vec", "MS2DeepScore", "DreaMS", "SIMBA"
])))
MODEL_COLORS = dict(zip(MODEL_NAMES, sns.color_palette("deep", n_colors=len(MODEL_NAMES))))


def plot_models(metric, selected, filename, ymax):
    fig, ax = plt.subplots(figsize=(5, 5))
    for name in selected:
        ax.plot(K_VALUES, accuracy_results[metric][0][name], marker="*",
                color=MODEL_COLORS[name], label=MODEL_LABELS[name])
    ax.set(xlabel="K candidates", ylabel="Precision", xlim=(0, 100), ylim=(0, ymax))
    ax.grid(True)
    ax.legend(loc="lower right")
    fig.savefig(FIGURE_FOLDER / (filename + SUFFIX + ".svg"), format="svg")
    plt.show()

### SIMBA precision at different normalized MCES similarity thresholds

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
for color, threshold in zip(sns.color_palette("deep", len(MIN_SIMILARITIES)), MIN_SIMILARITIES):
    y = accuracy_results["mces"][threshold][THRESHOLD_MODEL]
    ax.plot(K_VALUES, y, marker="*", linewidth=2, color=color,
            label=f"Max. normalized MCES distance = {1 - threshold:.2f}")
ax.set(xlabel="K candidates", ylabel="Precision", xlim=(min(K_VALUES), max(K_VALUES)), ylim=(0, 1))
ax.grid(True)
ax.legend(loc="lower right")
fig.savefig(FIGURE_FOLDER / ("ranking_different_similarities" + SUFFIX + ".svg"), format="svg")
plt.show()

### Model comparison: normalized MCES

In [ ]:
comparison_models = [name for name in MODEL_NAMES if name != "simba_20250422_only_massspecgym"]
plot_models("mces", comparison_models, "ranking_comparison", 0.70 if USE_GNPS else 0.65)

### SIMBA training-data comparison

In [ ]:
if not USE_GNPS:
    selected = ["simba_20250422_only_massspecgym", TARGET_SIMBA]
    fig, ax = plt.subplots(figsize=(5, 5))
    for name in selected:
        ax.plot(K_VALUES, accuracy_results["mces"][0][name], marker="*",
                color=MODEL_COLORS[name], label=MODEL_LABELS[name])
    ax.set(xlabel="K candidates", ylabel="Precision", ylim=(0, 0.65))
    ax.grid(True)
    ax.legend(loc="lower right")
    fig.savefig(FIGURE_FOLDER / "ranking_msg.svg", format="svg")
    fig.savefig(FIGURE_FOLDER / "ranking_msg.png", dpi=300)
    plt.show()
else:
    print("MassSpecGym comparison omitted: no separate MassSpecGym model in GNPS model list.")

### Model comparison: Tanimoto

In [ ]:
plot_models("tanimotos", comparison_models, "ranking_comparison_tanimotos", 0.55 if USE_GNPS else 0.45)